# AIC 2026 — Notebook B: Interactive Search, Clipping & VQA (LOCAL)

**Purpose**: Loads the pre-built FAISS and BM25 indices generated by **Notebook A on Kaggle**, performs multimodal event retrieval (KIS, VKIS, TRAKE), applies Unified Clipping with Reciprocal Rank Fusion (RRF), and answers questions via Gemini Flash VQA.

**Execution mode**: Local CPU. Runs interactively in Jupyter or as a Python script.

**Prerequisites**:
1. Download all 6 output files from Notebook A on Kaggle and place them in `kaggle_output/`.
2. Download `MobileCLIP2_S4.pt` and place it at `models/MobileCLIP2-S4/MobileCLIP2_S4.pt`.
3. Ensure your `.env` file contains `GEMINI_API_KEY_1=AIzaSy...` for VQA.

In [ ]:
# Cell 1: Package Installation
# Run once. Use the project virtual environment:
#   .venv\Scripts\pip install open_clip_torch faiss-cpu rank_bm25 google-generativeai matplotlib pillow pandas numpy python-dotenv
# NOTE: faiss-cpu (not faiss-gpu) for local CPU-only usage

In [ ]:
# Cell 2: Imports & Path Configuration
import os
import json
import pickle
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import faiss
from rank_bm25 import BM25Okapi
import open_clip
import google.generativeai as genai
from dotenv import load_dotenv

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(".")  # Assumes notebook is run from the AIHCM project root

# Indices downloaded from Kaggle Notebook A output
INDEX_DIR = PROJECT_ROOT / "kaggle_output"

# Local MobileCLIP2-S4 weights for query encoding
MODEL_WEIGHTS_PATH = PROJECT_ROOT / "models" / "MobileClipS2-4" / "MobileCLIP2_S4.pt"
MODEL_NAME = "MobileClipS2-4"

# Keyframe images (used for display)
KEYFRAMES_DIR = PROJECT_ROOT / "DAKE_output" / "extracted_keyframe_images"

assert INDEX_DIR.exists(), f"kaggle_output/ not found. Download Notebook A outputs first."
assert MODEL_WEIGHTS_PATH.exists(), f"Model not found at {MODEL_WEIGHTS_PATH}"

print(f"Loading indices from: {INDEX_DIR}")
print(f"Using model: {MODEL_NAME} from {MODEL_WEIGHTS_PATH}")

# ── Gemini API for VQA — loaded from local .env ───────────────────────────
load_dotenv(PROJECT_ROOT / ".env")
GEMINI_AVAILABLE = False
try:
    gemini_key = (
        os.getenv("GEMINI_API_KEY_1")
        or os.getenv("GEMINI_API_KEY")
    )
    if gemini_key:
        genai.configure(api_key=gemini_key)
        vqa_model = genai.GenerativeModel("gemini-2.0-flash")
        GEMINI_AVAILABLE = True
        print("Gemini API successfully configured for VQA!")
    else:
        print("Notice: No GEMINI_API_KEY found in .env. VQA will be disabled.")
except Exception as e:
    print(f"Notice: Gemini API setup failed ({e}). VQA disabled.")

In [ ]:
# Cell 3: Load Pre-built Indices & MobileCLIP2-S4 Query Encoder
print("Loading metadata...")
with open(INDEX_DIR / "metadata_df.pkl", "rb") as f:
    df_metadata = pickle.load(f)
print(f"Loaded {len(df_metadata)} keyframe records.")

print("Loading FAISS indices...")
visual_index = faiss.read_index(str(INDEX_DIR / "visual_faiss.index"))
text_index   = faiss.read_index(str(INDEX_DIR / "text_faiss.index"))

print("Loading BM25 corpus...")
with open(INDEX_DIR / "bm25_corpus.pkl", "rb") as f:
    bm25_data = pickle.load(f)
    bm25 = bm25_data["bm25"]
    bm25_corpus = bm25_data["corpus"]

# Load MobileCLIP2-S4 for query encoding — CPU only
device = torch.device("cpu")
print(f"Loading MobileCLIP2-S4 query encoder on {device}...")
clip_model, _, _ = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=str(MODEL_WEIGHTS_PATH)
)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
clip_model = clip_model.to(device).eval()  # .eval() important for BatchNorm layers

print("All indices and query encoder loaded successfully!")

In [ ]:
# Cell 4: Single Modality Search Functions
def encode_query_text(query_text):
    tokens = tokenizer([query_text]).to(device)
    with torch.no_grad():
        feats = clip_model.encode_text(tokens)
        feats /= feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)

def encode_query_image(image_path):
    _, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME, pretrained=str(MODEL_WEIGHTS_PATH))
    img = Image.open(image_path).convert("RGB")
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feats = clip_model.encode_image(tensor)
        feats /= feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy().astype(np.float32)


def search_visual(query_text=None, query_image_path=None, top_k=100):
    if query_image_path:
        q_emb = encode_query_image(query_image_path)
    elif query_text:
        q_emb = encode_query_text(query_text)
    else:
        raise ValueError("Must provide text or image query for visual search.")
    scores, indices = visual_index.search(q_emb, top_k)
    results = []
    for idx, sc in zip(indices[0], scores[0]):
        if idx < len(df_metadata):
            row = df_metadata.iloc[idx].to_dict()
            row['score'] = float(sc)
            row['source'] = 'visual'
            results.append(row)
    return results


def search_text_emb(query_text, top_k=100):
    q_emb = encode_query_text(query_text)
    scores, indices = text_index.search(q_emb, top_k)
    results = []
    for idx, sc in zip(indices[0], scores[0]):
        if idx < len(df_metadata):
            row = df_metadata.iloc[idx].to_dict()
            row['score'] = float(sc)
            row['source'] = 'text_emb'
            results.append(row)
    return results


def search_bm25(query_text, top_k=100):
    tokens = query_text.lower().split()
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(-scores)[:top_k]
    results = []
    for idx in top_indices:
        if scores[idx] > 0 and idx < len(df_metadata):
            row = df_metadata.iloc[idx].to_dict()
            row['score'] = float(scores[idx])
            row['source'] = 'bm25'
            results.append(row)
    return results

In [ ]:
# Cell 5: Score Fusion Engine (RRF & Weighted Fallback)
def reciprocal_rank_fusion(results_lists, k=60):
    """Scale-free RRF algorithm: score(doc) = sum(1 / (k + rank))"""
    rrf_scores = {}
    item_map = {}

    for res_list in results_lists:
        for rank, item in enumerate(res_list):
            key = f"{item['video_name']}_{item['frame_idx']}"
            if key not in item_map:
                item_map[key] = item
                rrf_scores[key] = 0.0
            rrf_scores[key] += 1.0 / (k + rank + 1)

    sorted_keys = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)
    fused = []
    for k_val in sorted_keys:
        item = item_map[k_val].copy()
        item['score'] = rrf_scores[k_val]
        item['fusion_method'] = 'RRF'
        fused.append(item)
    return fused


def weighted_fusion(results_lists, weights=(0.4, 0.3, 0.3)):
    """Fallback weighted sum with min-max normalization per list"""
    weighted_scores = {}
    item_map = {}

    for idx, res_list in enumerate(results_lists):
        w = weights[idx]
        if not res_list:
            continue
        scores = [it['score'] for it in res_list]
        min_s, max_s = min(scores), max(scores)
        denom = max_s - min_s if max_s > min_s else 1.0

        for item in res_list:
            key = f"{item['video_name']}_{item['frame_idx']}"
            if key not in item_map:
                item_map[key] = item
                weighted_scores[key] = 0.0
            norm_score = (item['score'] - min_s) / denom
            weighted_scores[key] += w * norm_score

    sorted_keys = sorted(weighted_scores.keys(), key=lambda x: weighted_scores[x], reverse=True)
    fused = []
    for k_val in sorted_keys:
        item = item_map[k_val].copy()
        item['score'] = weighted_scores[k_val]
        item['fusion_method'] = 'Weighted'
        fused.append(item)
    return fused


def retrieve_all(query_text=None, query_image_path=None, top_k=100, mode="rrf"):
    results = []
    if query_text or query_image_path:
        results.append(search_visual(query_text, query_image_path, top_k))
    if query_text:
        results.append(search_text_emb(query_text, top_k))
        results.append(search_bm25(query_text, top_k))

    if mode == "rrf":
        return reciprocal_rank_fusion(results)
    else:
        return weighted_fusion(results)


In [ ]:
# Cell 6: Unified Clipping Algorithm (Two-pointer + NMS)
def unified_clipping(candidate_frames, max_clip_duration=20.0, top_k_clips=20, nms_threshold=0.5):
    """
    Linear two-pointer sliding window sweep over frame candidates grouped by video.
    Ranked by highest frame score. Applies NMS to eliminate duplicate overlapping windows.
    """
    by_video = {}
    for f in candidate_frames:
        v = f['video_name']
        if v not in by_video:
            by_video[v] = []
        by_video[v].append(f)

    clips = []
    for v_name, frames in by_video.items():
        frames = sorted(frames, key=lambda x: x['pts_time'])
        n = len(frames)
        left = 0
        for right in range(n):
            while frames[right]['pts_time'] - frames[left]['pts_time'] > max_clip_duration:
                left += 1
            window = frames[left:right+1]
            best_frame = max(window, key=lambda x: x['score'])
            clips.append({
                "video_name": v_name,
                "start_time": window[0]['pts_time'],
                "end_time": window[-1]['pts_time'],
                "duration": window[-1]['pts_time'] - window[0]['pts_time'],
                "best_frame": best_frame,
                "score": best_frame['score'],
                "n_frames": len(window)
            })

    clips = sorted(clips, key=lambda x: x['score'], reverse=True)

    # NMS Deduplication
    selected_clips = []
    for c in clips:
        overlap = False
        for sel in selected_clips:
            if sel['video_name'] == c['video_name']:
                inter_start = max(sel['start_time'], c['start_time'])
                inter_end   = min(sel['end_time'],   c['end_time'])
                if inter_end > inter_start:
                    inter_dur = inter_end - inter_start
                    union_dur = (sel['end_time'] - sel['start_time']) + (c['end_time'] - c['start_time']) - inter_dur
                    if inter_dur / max(1e-5, union_dur) > nms_threshold:
                        overlap = True
                        break
        if not overlap:
            selected_clips.append(c)
            if len(selected_clips) >= top_k_clips:
                break

    return selected_clips

In [ ]:
# Cell 7: Gemini Flash VQA Integration
def answer_vqa(image_path, question_text):
    if not GEMINI_AVAILABLE:
        return "Gemini API key not configured. Cannot perform VQA."
    try:
        img = Image.open(image_path)
        prompt = f"Hãy trả lời câu hỏi sau một cách ngắn gọn, chính xác bằng tiếng Việt dựa vào hình ảnh này:\nCâu hỏi: {question_text}"
        response = vqa_model.generate_content([img, prompt])
        return response.text.strip()
    except Exception as e:
        return f"VQA Error: {e}"

In [ ]:
# Cell 8: Result Visualization Engine
def show_results(clips, max_display=10, cols=5):
    display_clips = clips[:max_display]
    if not display_clips:
        print("No matching clips found.")
        return

    rows = (len(display_clips) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1 or cols == 1:
        axes = np.atleast_2d(axes)

    for idx, c in enumerate(display_clips):
        r, col = idx // cols, idx % cols
        ax = axes[r, col]
        bf = c['best_frame']

        # Try the path stored in metadata first; fall back to local keyframes dir
        img_p = Path(bf['image_path'])
        if not img_p.exists():
            img_p = KEYFRAMES_DIR / bf['video_name'] / f"{bf['n']:04d}.jpg"

        if img_p.exists():
            ax.imshow(Image.open(img_p))
        else:
            ax.text(0.5, 0.5, "Image Missing", ha='center', va='center')

        title = f"#{idx+1}: {c['video_name']}\n{c['start_time']:.1f}s-{c['end_time']:.1f}s | Score: {c['score']:.4f}"
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    for i in range(idx + 1, rows * cols):
        r, col = i // cols, i % cols
        axes[r, col].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 9: Interactive KIS Search Demo
query = "con tôm hấp sốt mayonnaise"
print(f"Executing search for query: '{query}'...")

raw_candidates = retrieve_all(query, top_k=50, mode="rrf")
top_clips = unified_clipping(raw_candidates, max_clip_duration=20.0, top_k_clips=10)

print(f"Found {len(top_clips)} video event clips after unified clipping and NMS:")
show_results(top_clips)

In [ ]:
# Cell 10: TRAKE Temporal Search & VQA Demo
print("=== TRAKE Multi-Query Temporal Alignment Demo ===")
multi_queries = [
    "chuẩn bị nguyên liệu tôm",
    "hấp tôm trên nồi",
    "món ăn hoàn thành bày ra đĩa"
]

trake_results = []
for q_idx, q in enumerate(multi_queries):
    cands = retrieve_all(q, top_k=30, mode="rrf")
    for c in cands:
        c['query_idx'] = q_idx
    trake_results.extend(cands)

trake_clips = unified_clipping(trake_results, max_clip_duration=60.0, top_k_clips=5)
show_results(trake_clips)

if trake_clips and GEMINI_AVAILABLE:
    top_frame = trake_clips[0]['best_frame']['image_path']
    question = "Trong hình có những món ăn nào?"
    answer = answer_vqa(top_frame, question)
    print(f"\n[VQA Demo] Question: {question}")
    print(f"[VQA Demo] Answer: {answer}")